In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle
import os

# Load the data we saved in Phase 2
all_data = pd.read_csv('../data/processed/all_data.csv')

print(f"Loaded {len(all_data)} rows")
print(f"Crash rows:     {(all_data['label'] == 1).sum()}")
print(f"Non-crash rows: {(all_data['label'] == 0).sum()}")

Loaded 137095 rows
Crash rows:     10795
Non-crash rows: 126300


In [2]:
def extract_windows(df, window_size=100, step=50):
    """
    Cuts the sensor stream into overlapping 2-second windows.
    window_size = 100 samples = 2 seconds at 50Hz
    step = 50 samples = 1 second (50% overlap)
    """
    windows = []
    labels  = []

    # Extract just the 6 sensor columns as a numpy array
    data      = df[['acc_x','acc_y','acc_z',
                     'gyro_x','gyro_y','gyro_z']].values
    label_col = df['label'].values

    for start in range(0, len(data) - window_size, step):
        end    = start + window_size
        window = data[start:end]          # shape: [100, 6]

        # Label this window as crash if more than
        # 50% of its rows are crash rows
        window_label = int(label_col[start:end].mean() >= 0.5)

        windows.append(window)
        labels.append(window_label)

    windows = np.array(windows)   # shape: [num_windows, 100, 6]
    labels  = np.array(labels)    # shape: [num_windows]

    return windows, labels


windows, labels = extract_windows(all_data)

print(f"Total windows:      {len(windows)}")
print(f"Crash windows:      {labels.sum()}")
print(f"Non-crash windows:  {(labels == 0).sum()}")
print(f"Window shape:       {windows[0].shape}  ← should be (100, 6)")

Total windows:      2740
Crash windows:      215
Non-crash windows:  2525
Window shape:       (100, 6)  ← should be (100, 6)


In [3]:
def augment_crash_windows(crash_windows, n_augmentations=4):
    """
    Creates artificial variations of crash windows
    to increase crash training data.
    Only applied to crash windows — never to non-crash.
    """
    augmented = []

    for window in crash_windows:
        for _ in range(n_augmentations):
            aug = window.copy()

            # Randomly apply one or more augmentations
            if np.random.rand() > 0.5:
                # Time warp: stretch or compress time axis slightly
                factor      = np.random.uniform(0.9, 1.1)
                old_indices = np.arange(len(aug))
                new_indices = np.linspace(0, len(aug)-1,
                                          int(len(aug)*factor))
                for axis in range(aug.shape[1]):
                    aug[:, axis] = np.interp(
                        np.linspace(0, len(new_indices)-1, len(aug)),
                        np.arange(len(new_indices)),
                        np.interp(new_indices, old_indices, aug[:, axis])
                    )

            if np.random.rand() > 0.5:
                # Magnitude jitter: add tiny random noise
                aug += np.random.normal(0, 0.05, aug.shape)

            if np.random.rand() > 0.5:
                # Axis swap: simulate phone in different orientation
                aug[:, 0], aug[:, 1] = aug[:, 1].copy(), aug[:, 0].copy()
                aug[:, 3], aug[:, 4] = aug[:, 4].copy(), aug[:, 3].copy()

            augmented.append(aug)

    return np.array(augmented)


# Separate crash and non-crash windows
crash_windows     = windows[labels == 1]
non_crash_windows = windows[labels == 0]

print(f"Original crash windows:     {len(crash_windows)}")

# Augment crash windows only
augmented_crashes = augment_crash_windows(crash_windows,
                                          n_augmentations=4)

print(f"Augmented crash windows:    {len(augmented_crashes)}")

# Combine original + augmented crashes with all non-crash
X = np.concatenate([
    crash_windows,
    augmented_crashes,
    non_crash_windows
])

y = np.concatenate([
    np.ones(len(crash_windows)),
    np.ones(len(augmented_crashes)),
    np.zeros(len(non_crash_windows))
])

print(f"\nFinal dataset:")
print(f"Total windows:     {len(X)}")
print(f"Crash windows:     {int(y.sum())}")
print(f"Non-crash windows: {int((y == 0).sum())}")

Original crash windows:     215
Augmented crash windows:    860

Final dataset:
Total windows:     3600
Crash windows:     1075
Non-crash windows: 2525


In [4]:
def compute_features(window):
    """
    Computes a flat feature vector from one window.
    The Random Forest needs flat numbers, not a 2D array.
    The 1D-CNN will use the raw window directly.
    """
    features = []

    # Per-axis stats for all 6 axes
    for i in range(6):
        axis = window[:, i]
        features += [
            np.mean(axis),                      # average value
            np.std(axis),                       # how much it varies
            np.min(axis),                       # lowest point
            np.max(axis),                       # highest point
            np.sqrt(np.mean(axis**2))           # RMS energy
        ]

    # Resultant magnitude (overall movement strength)
    acc_mag  = np.sqrt(window[:,0]**2 +
                       window[:,1]**2 +
                       window[:,2]**2)
    gyro_mag = np.sqrt(window[:,3]**2 +
                       window[:,4]**2 +
                       window[:,5]**2)

    features += [np.mean(acc_mag),
                 np.max(acc_mag),
                 np.std(acc_mag)]
    features += [np.mean(gyro_mag),
                 np.max(gyro_mag),
                 np.std(gyro_mag)]

    # Peak G-force and jerk (rate of change)
    features.append(np.max(acc_mag))
    jerk = np.diff(acc_mag)
    features.append(np.max(np.abs(jerk)))

    return np.array(features)


# Apply to all windows
print("Computing features... (may take a minute)")
X_features = np.array([compute_features(w) for w in X])

print(f"Feature matrix shape: {X_features.shape}")
print(f"  ↑ means {X_features.shape[0]} windows")
print(f"    each with {X_features.shape[1]} features")

Computing features... (may take a minute)
Feature matrix shape: (3600, 38)
  ↑ means 3600 windows
    each with 38 features


In [5]:
# Normalise features (zero mean, unit variance)
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_features)

# Save scaler — needed later for the Flutter app
os.makedirs('../models', exist_ok=True)
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Scaler saved.")

# Split: 70% train, 15% validation, 15% test
# stratify=y ensures equal crash/non-crash ratio in each split
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X_scaled, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp,
    test_size=0.50,
    random_state=42,
    stratify=y_tmp
)

print(f"\nSplit summary:")
print(f"Train:      {len(X_tr)} windows")
print(f"Validation: {len(X_val)} windows")
print(f"Test:       {len(X_test)} windows")

Scaler saved.

Split summary:
Train:      2520 windows
Validation: 540 windows
Test:       540 windows


In [6]:
# Save raw windows for the CNN (Phase 5)
# Save feature vectors for the Random Forest (Phase 4)

# Split raw windows using same indices
X_raw_tr, X_raw_tmp, _, _ = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)
X_raw_val, X_raw_test, _, _ = train_test_split(
    X_raw_tmp, _,
    test_size=0.50,
    random_state=42,
    stratify=_
)

np.save('../data/processed/X_train_features.npy', X_tr)
np.save('../data/processed/X_val_features.npy',   X_val)
np.save('../data/processed/X_test_features.npy',  X_test)
np.save('../data/processed/y_train.npy',           y_tr)
np.save('../data/processed/y_val.npy',             y_val)
np.save('../data/processed/y_test.npy',            y_test)
np.save('../data/processed/X_train_raw.npy',       X_raw_tr)
np.save('../data/processed/X_val_raw.npy',         X_raw_val)
np.save('../data/processed/X_test_raw.npy',        X_raw_test)

print("All data saved and ready for Phase 4!")

All data saved and ready for Phase 4!
